# HealthPulse AI 360 — J2 : Acquisition des données officielles sur le diabète

## Objectif
Récupérer les premières données officielles de
l'Assurance Maladie à l'aide de Python.

## Source
- Organisme : Assurance Maladie (CNAM)
- Portail : Data ameli
- Jeu de données : Effectifs par pathologie
- Période disponible : 2015–2024
- Échelle : département et région

## Méthodologie
1. Tester l'API.
2. Importer les données.
3. Explorer les colonnes.
4. Contrôler la qualité.
5. Sauvegarder les données brutes.


In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

In [2]:
## Importation des librairies

import pandas as pd
import requests
from pathlib import Path


In [3]:
## Définir les chemins
projet = Path(r"D:\HealthPulse_AI_360")

dossier_raw = projet / "data" / "raw" / "diabete"

dossier_raw.mkdir(parents=True, exist_ok=True)

print("Dossier :", dossier_raw)


Dossier : D:\HealthPulse_AI_360\data\raw\diabete


In [4]:
#Tester l'API du site


url = (
    "https://data.ameli.fr/api/explore/v2.1/"
    "catalog/datasets/effectifs/records"
)

params = {
    "limit": 5
}

reponse = requests.get(
    url,
    params=params,
    timeout=30
)

print("Code HTTP :", reponse.status_code)
print("URL :", reponse.url)

reponse.raise_for_status()


Code HTTP : 200
URL : https://data.ameli.fr/api/explore/v2.1/catalog/datasets/effectifs/records?limit=5


In [5]:
#Lire la réponse JSON


donnees = reponse.json()

print(donnees.keys())


dict_keys(['total_count', 'results'])


In [6]:

df = pd.DataFrame(donnees["results"])

df.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,ntop,npop,prev,niveau_prioritaire,libelle_classe_age,libelle_sexe,tri
0,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,01,971,280,17040,1.643,"[2, 3]",de 50 à 54 ans,femmes,48.0
1,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,03,999,50,6090,0.771,"[2, 3]",de 50 à 54 ans,femmes,48.0
2,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,04,974,390,32630,1.205,"[2, 3]",de 50 à 54 ans,femmes,48.0
3,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,11,77,350,48230,0.732,"[2, 3]",de 50 à 54 ans,femmes,48.0
4,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,11,91,400,43370,0.927,"[2, 3]",de 50 à 54 ans,femmes,48.0


In [7]:
#Découvrir les colonnes réelles


print("Dimensions :", df.shape)

print("Colonnes :")
print(df.columns.tolist())


Dimensions : (5, 16)
Colonnes :
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5', 'sexe', 'region', 'dept', 'ntop', 'npop', 'prev', 'niveau_prioritaire', 'libelle_classe_age', 'libelle_sexe', 'tri']


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   annee               5 non-null      object 
 1   patho_niv1          5 non-null      object 
 2   patho_niv2          5 non-null      object 
 3   patho_niv3          5 non-null      object 
 4   top                 5 non-null      object 
 5   cla_age_5           5 non-null      object 
 6   sexe                5 non-null      object 
 7   region              5 non-null      object 
 8   dept                5 non-null      object 
 9   ntop                5 non-null      int64  
 10  npop                5 non-null      int64  
 11  prev                5 non-null      float64
 12  niveau_prioritaire  5 non-null      object 
 13  libelle_classe_age  5 non-null      object 
 14  libelle_sexe        5 non-null      object 
 15  tri                 5 non-null      float64
dtypes: float64(2

In [9]:
#Contrôles de qualité


print("Nombre de lignes :", len(df))

print("Valeurs manquantes :")
print(df.isna().sum())

print("Doublons complets :", df.duplicated().sum())


Nombre de lignes : 5
Valeurs manquantes :
annee                 0
patho_niv1            0
patho_niv2            0
patho_niv3            0
top                   0
cla_age_5             0
sexe                  0
region                0
dept                  0
ntop                  0
npop                  0
prev                  0
niveau_prioritaire    0
libelle_classe_age    0
libelle_sexe          0
tri                   0
dtype: int64


TypeError: unhashable type: 'list'

In [10]:
#Enregistrer l'échantillon brut


fichier = dossier_raw / "ameli_echantillon_j02.json"

fichier.write_text(
    reponse.text,
    encoding="utf-8"
)

print("Fichier enregistré :", fichier)


Fichier enregistré : D:\HealthPulse_AI_360\data\raw\diabete\ameli_echantillon_j02.json


In [11]:
#Enregistrer le fichier en CSV
df.to_csv(
    dossier_raw / "ameli_echantillon_j02.csv",
    index=False
)



## Bilan du J2

### Objectif
Découvrir et interroger les données publiques
de l'Assurance Maladie.

### Travail réalisé
- Identification de la source officielle.
- Première requête HTTP.
- Lecture d'une réponse JSON.
- Création d'un DataFrame Pandas.
- Exploration des colonnes.
- Contrôles de qualité.
- Sauvegarde des données brutes.

### Difficultés rencontrées
À compléter avec mes propres observations.

### Interprétation
Les données représentent des informations
agrégées sur les personnes prises en charge.
Elles ne correspondent pas nécessairement
à tous les cas de diabète dans la population.

### Compétences acquises
- Requests et API REST.
- Lecture du JSON.
- Pandas et exploration des données.
- Pathlib et gestion des fichiers.

### Prochaine étape
Identifier et extraire un jeu de données
départemental sur le diabète.
